<a href="https://colab.research.google.com/github/Kaneriah43/Flyrank_Ml_Internship/blob/main/work/notebooks/w04_signal_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Kaneriah43/Flyrank_Ml_Internship/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Distributions

*Look before deciding: distributions of your key fields. Note the heavy tails.*

Key fields show heavy right tails — a small number of pages dominate impressions and clicks. Most pages cluster near zero. Median is a more honest threshold than mean for this reason.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [1]:
import os
import getpass

HF_TOKEN = os.environ.get("HF_TOKEN")

if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass

HF_TOKEN = HF_TOKEN or getpass.getpass(
    "Paste your Hugging Face READ token (hf_...): ")

Paste your Hugging Face READ token (hf_...): ··········


In [2]:
import duckdb

con = duckdb.connect()

con.execute(
    f"CREATE OR REPLACE SECRET hf "
    f"(TYPE huggingface, TOKEN '{HF_TOKEN}')")

In [3]:
REL = "hf://datasets/FlyRank/internship-warehouse"

In [4]:
TABLES = {
    "dim_clients":    f"read_parquet('{REL}/dim_clients.parquet')",
    "dim_content":    f"read_parquet('{REL}/dim_content.parquet')",
    "fact_daily":     f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    "fact_query_90d": f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

In [6]:
import numpy as np
df = con.execute(f"""
    WITH base AS (
        SELECT
            d.content_hash_id,
            d.client_hash_id,
            d.month,
            d.report_date,
            d.gsc_impressions,
            d.gsc_clicks,
            d.gsc_avg_position,
            MAX(d.report_date) OVER (
                PARTITION BY d.content_hash_id, d.client_hash_id
            ) AS max_date
        FROM {TABLES['fact_daily']} d
        WHERE d.month = '2026-03'
          AND d.gsc_data_available = TRUE
    )
    SELECT
        b.content_hash_id,
        b.client_hash_id,
        b.month,
        SUM(b.gsc_impressions)                                       AS impressions_90d,
        SUM(b.gsc_clicks)                                            AS clicks_90d,
        AVG(b.gsc_avg_position)                                      AS avg_position,
        SUM(b.gsc_clicks) / NULLIF(SUM(b.gsc_impressions), 0)       AS ctr,
        SUM(CASE WHEN b.report_date >= (b.max_date - INTERVAL 30 DAYS)
                 THEN b.gsc_impressions ELSE 0 END)                  AS impressions_last30,
        SUM(CASE WHEN b.report_date < (b.max_date - INTERVAL 30 DAYS)
                 THEN b.gsc_impressions ELSE 0 END)                  AS impressions_prev30,
        c.content_type,
        c.word_count,
        c.last_optimized_date,
        DATEDIFF('day', c.last_optimized_date, MAX(b.report_date))  AS days_since_last_update,
        DATEDIFF('day', c.content_created_date, MAX(b.report_date)) AS content_age_days
    FROM base b
    JOIN {TABLES['dim_content']} c
        ON b.content_hash_id = c.content_hash_id
    GROUP BY
        b.content_hash_id, b.client_hash_id, b.month,
        c.content_type, c.word_count,
        c.last_optimized_date, c.content_created_date
""").df()

# Derive trend_direction
df["trend_direction"] = np.where(
    df["impressions_last30"] > df["impressions_prev30"], "up",
    np.where(df["impressions_last30"] < df["impressions_prev30"], "down", "flat")
)

print(f"Rows loaded: {len(df):,}")
df.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows loaded: 176,738


,content_hash_id,client_hash_id,month,impressions_90d,clicks_90d,avg_position,ctr,impressions_last30,impressions_prev30,content_type,word_count,last_optimized_date,days_since_last_update,content_age_days,trend_direction
0,content_14a86c63a214f648,client_2094c6eb080311d5,2026-03,44.0,0.0,30.479101,0.000000,44.0,0.0,keyword article,2949,NaT,<NA>,112,up
1,content_14b1a02c1b8557fb,client_2094c6eb080311d5,2026-03,90.0,0.0,28.426940,0.000000,90.0,0.0,keyword article,4279,NaT,<NA>,108,up
2,content_14c17f59aa610ab3,client_2094c6eb080311d5,2026-03,122.0,2.0,5.882498,0.016393,122.0,0.0,keyword article,3277,2026-06-22,-83,42,up
3,content_153357ef5824c7cc,client_2094c6eb080311d5,2026-03,9.0,0.0,41.700000,0.000000,9.0,0.0,keyword article,2988,NaT,<NA>,19,up
4,content_15565677b6e1792f,client_2094c6eb080311d5,2026-03,148.0,1.0,12.350803,0.006757,148.0,0.0,keyword article,3011,NaT,<NA>,20,up


In [ ]:
numeric_fields = ["impressions_90d", "clicks_90d", "days_since_last_update",
                  "ctr", "avg_position", "content_age_days",
                  "impressions_last30", "impressions_prev30"]

for col in numeric_fields:
    print(f"=== {col} ===")
    print(df[col].describe().round(3))
    print(f"  p90: {df[col].quantile(0.90):.1f}")
    print(f"  p99: {df[col].quantile(0.99):.1f}")
    print()

## 2. Signal test #1 / #2 / #3 (verdict each)

*Three safe signals, each with a mini-test and a verdict: CONFIRMED / OPPOSITE / MIXED / FALSE.*

Testing three signals my rule relies on — staleness, CTR vs position, and trend direction vs impressions. Each gets a bucket table, row count, and a one-word verdict.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [8]:
# Signal 1 — Staleness vs CTR
import pandas as pd

print("=== SIGNAL 1: Staleness vs CTR ===")
bins   = [0, 30, 90, 180, 365, 99999]
labels = ["<30d", "30-90d", "90-180d", "180-365d", "365d+"]
df["staleness_bucket"] = pd.cut(
    df["days_since_last_update"], bins=bins, labels=labels
)

t1 = df.groupby("staleness_bucket").agg(
    n=("content_hash_id", "count"),
    avg_ctr=("ctr", "mean"),
    pct_declining=("trend_direction", lambda x: (x == "down").mean())
).round(3)
print(t1)
print("Verdict: [CONFIRMED / OPPOSITE / MIXED / FALSE]")
print("Reason:  [fill after seeing output]\n")

# Signal 2 — CTR vs avg_position bucket
print("=== SIGNAL 2: CTR vs Position ===")
df["position_bucket"] = pd.cut(
    df["avg_position"],
    bins=[0, 3, 5, 10, 20, 99999],
    labels=["1-3", "3-5", "5-10", "10-20", "20+"]
)

t2 = df.groupby("position_bucket").agg(
    n=("content_hash_id", "count"),
    avg_ctr=("ctr", "mean"),
    avg_impressions=("impressions_90d", "mean")
).round(3)
print(t2)
print("Verdict: [CONFIRMED / OPPOSITE / MIXED / FALSE]")
print("Reason:  [fill after seeing output]\n")

# Signal 3 — Trend direction vs impressions
print("=== SIGNAL 3: Trend Direction vs Impressions ===")
t3 = df.groupby("trend_direction").agg(
    n=("content_hash_id", "count"),
    avg_impressions=("impressions_90d", "mean"),
    avg_ctr=("ctr", "mean")
).round(3)
print(t3)
print("Verdict: [CONFIRMED / OPPOSITE / MIXED / FALSE]")
print("Reason:  [fill after seeing output]")

=== SIGNAL 1: Staleness vs CTR ===
                  n  avg_ctr  pct_declining
staleness_bucket                           
<30d              0      NaN            NaN
30-90d            0      NaN            NaN
90-180d           0      NaN            NaN
180-365d          0      NaN            NaN
365d+             0      NaN            NaN
Verdict: [CONFIRMED / OPPOSITE / MIXED / FALSE]
Reason:  [fill after seeing output]

=== SIGNAL 2: CTR vs Position ===
                     n  avg_ctr  avg_impressions
position_bucket                                 
1-3              16144    0.011         2435.859
3-5              26593    0.006         2797.953
5-10             55395    0.004         1313.858
10-20            32203    0.003         1081.532
20+              44969    0.002         1319.016
Verdict: [CONFIRMED / OPPOSITE / MIXED / FALSE]
Reason:  [fill after seeing output]

=== SIGNAL 3: Trend Direction vs Impressions ===
                      n  avg_impressions  avg_ctr
trend_direc

/tmp/ipykernel_423/3346372468.py:11: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  t1 = df.groupby("staleness_bucket").agg(
/tmp/ipykernel_423/3346372468.py:28: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  t2 = df.groupby("position_bucket").agg(


## 3. The flag-linked test

*Pick a signal one of FlyRank's real flags relies on. Does the data support the rule's assumption?*

FlyRank's refresh flag relies on staleness — the assumption is that pages not updated in 180+ days show declining impressions relative to fresher pages at the same position. Testing that directly here.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [9]:
print("=== FLAG-LINKED TEST: Staleness → impression drop ===")

stale  = df["days_since_last_update"] >= 180

result = df.groupby([
    "position_bucket",
    stale.rename("is_stale")
]).agg(
    n=("content_hash_id", "count"),
    avg_impressions=("impressions_90d", "mean"),
    avg_ctr=("ctr", "mean"),
    pct_declining=("trend_direction", lambda x: (x == "down").mean())
).round(3).unstack("is_stale")

print(result)
print(f"\nTotal stale pages (180d+): {stale.sum():,}")
print(f"Stale as % of total:       {stale.mean()*100:.1f}%")
print("\nDoes the data support FlyRank's refresh flag assumption?")
print("[Fill your answer here after seeing the output]")

=== FLAG-LINKED TEST: Staleness → impression drop ===
                     n avg_impressions avg_ctr pct_declining
is_stale         False           False   False         False
position_bucket                                             
1-3               2736        8410.500   0.004           0.0
3-5               7279        7305.385   0.004           0.0
5-10             14489        3736.009   0.003           0.0
10-20             8960        2527.402   0.003           0.0
20+               6277        5617.519   0.002           0.0

Total stale pages (180d+): 0
Stale as % of total:       0.0%

Does the data support FlyRank's refresh flag assumption?
[Fill your answer here after seeing the output]


/tmp/ipykernel_423/2625963090.py:5: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  result = df.groupby([


## 4. What this means in practice

*Two or three sentences: what a content team should take from this.*

The data [confirms / partially confirms / does not confirm] that staleness predicts declining impressions. Stale pages show [X]% lower average impressions than fresh pages at the same position bucket — meaning the refresh flag assumption holds [strongly / weakly / only for certain position ranges].

Heavy tails in impressions mean a small number of pages drive most of the signal. A content team should focus refresh effort on high-impression stale pages first — the CTR gap is largest there and the traffic impact is highest.

One caveat: correlation is not causation. A stale page may have declining impressions due to topic decay, not just content age. Refreshing format without refreshing topic relevance may not recover performance.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.